# Caspase Activation Across All Cell Lines

Simple analysis comparing caspase activation (Active/Inactive) across **14 different cell lines**.

**Datasets**:
- AID_654 through AID_667 (13 cell types from NCATS)
- concat_hepg2_cleaned.csv (HepG2 combined)
- AID_1346980_cleaned.csv (Caspase Activation)

In [310]:
import pandas as pd
import numpy as np

print("Libraries loaded!")

Libraries loaded!


In [311]:
# Step 1: Load raw data
df = pd.read_csv("/Users/yaminichitikela/Documents/casepase/Casepase-activation/notebooks/data_merging/filtered_combined_all_caspase_cell_to_one_casepase.csv")
print(df.shape)
df.head()

(31386, 7)


,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,source
0,5009,144212021,4,CC(CN)O,Inactive,0,combined_hepg2_casepase.csv
1,5009,144212021,4,CC(CN)O,Inactive,0,AID_1346980
2,2142,144209154,6,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,Inactive,0,AID_1346980
3,2142,144209154,6,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,Inconclusive,10,combined_hepg2_casepase.csv
4,802,17389930,11,C(CCl)Cl,Inactive,0,AID_658


In [312]:
# Step 2: Keep only Active, Inactive, Inconclusive
df = df[df["PUBCHEM_ACTIVITY_OUTCOME"].isin(["Active", "Inactive", "Inconclusive"])].copy()
print(df["PUBCHEM_ACTIVITY_OUTCOME"].value_counts())

PUBCHEM_ACTIVITY_OUTCOME
Inactive        29893
Inconclusive      970
Active            523
Name: count, dtype: int64


In [313]:
# Step 3: Pivot — one row per CID, one column per cell line
# Sort so Active > Inconclusive > Inactive (aggfunc=first picks most informative)
priority = {"Active": 0, "Inconclusive": 1, "Inactive": 2}
df["_p"] = df["PUBCHEM_ACTIVITY_OUTCOME"].map(priority)
df = df.sort_values("_p").drop(columns="_p")

compound_matrix = df.pivot_table(
    index="PUBCHEM_CID",
    columns="source",
    values="PUBCHEM_ACTIVITY_OUTCOME",
    aggfunc="first"
).reset_index()

compound_matrix.columns.name = None  # remove the "source" axis label
print(compound_matrix.shape)
compound_matrix.head()

(7716, 15)


,PUBCHEM_CID,AID_1346980,AID_655,AID_656,AID_657,AID_658,AID_659,AID_660,AID_661,AID_663,AID_664,AID_665,AID_666,AID_667,combined_hepg2_casepase.csv
0,4,Inactive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inactive
1,6,Inactive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inconclusive
2,11,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive
3,13,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive
4,33,Inactive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inactive


In [314]:
# Step 4: Build pattern string for each CID + add behavior label
source_cols = [c for c in compound_matrix.columns if c != "PUBCHEM_CID"]

def get_pattern(row):
    parts = [f"{col}={row[col]}" for col in source_cols if pd.notna(row[col])]
    return " | ".join(parts)

def get_behavior(row):
    vals = row[source_cols].dropna().unique()
    if len(vals) == 0:
        return "Unknown"
    elif len(vals) == 1:
        return vals[0]
    else:
        return "Mixed (" + " + ".join(sorted(vals)) + ")"

compound_matrix["pattern"]  = compound_matrix.apply(get_pattern, axis=1)
compound_matrix["behavior"] = compound_matrix.apply(get_behavior, axis=1)

print(compound_matrix["behavior"].value_counts())
compound_matrix.head()

behavior
Inactive                                    6967
Mixed (Inactive + Inconclusive)              359
Mixed (Active + Inactive)                    110
Mixed (Active + Inactive + Inconclusive)      77
Mixed (Active + Inconclusive)                 70
Inconclusive                                  67
Active                                        66
Name: count, dtype: int64


,PUBCHEM_CID,AID_1346980,AID_655,AID_656,AID_657,AID_658,AID_659,AID_660,AID_661,AID_663,AID_664,AID_665,AID_666,AID_667,combined_hepg2_casepase.csv,pattern,behavior
0,4,Inactive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inactive,AID_1346980=Inactive | combined_hepg2_casepase...,Inactive
1,6,Inactive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inconclusive,AID_1346980=Inactive | combined_hepg2_casepase...,Mixed (Inactive + Inconclusive)
2,11,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,AID_1346980=Inactive | AID_655=Inactive | AID_...,Inactive
3,13,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,Inactive,AID_1346980=Inactive | AID_655=Inactive | AID_...,Inactive
4,33,Inactive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Inactive,AID_1346980=Inactive | combined_hepg2_casepase...,Inactive


In [315]:
# Step 5: consistent_groups = ALL CIDs sorted by pattern
# Compounds sharing the same pattern are grouped together
consistent_groups = compound_matrix.copy()

# Count how many CIDs share each pattern (for sorting largest groups first)
pattern_size = consistent_groups.groupby("pattern")["PUBCHEM_CID"].transform("count")
consistent_groups["pattern_size"] = pattern_size

consistent_groups = consistent_groups.sort_values(
    by=["pattern_size", "pattern", "PUBCHEM_CID"],
    ascending=[False, True, True]
).drop(columns="pattern_size").reset_index(drop=True)

# Save
consistent_groups.to_csv(
    "/Users/yaminichitikela/Documents/casepase/Casepase-activation/results/all_cell_line_comparisions/consistent_compounds_caspase_all.csv",
    index=False
)
print(f"Saved {len(consistent_groups)} CIDs sorted by pattern")

# Verify CID 299
consistent_groups[consistent_groups["PUBCHEM_CID"] == 299]

Saved 7716 CIDs sorted by pattern


,PUBCHEM_CID,AID_1346980,AID_655,AID_656,AID_657,AID_658,AID_659,AID_660,AID_661,AID_663,AID_664,AID_665,AID_666,AID_667,combined_hepg2_casepase.csv,pattern,behavior
7703,299,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Active,Inconclusive,Inactive,Inconclusive,Active,AID_1346980=Inconclusive | AID_655=Inconclusiv...,Mixed (Active + Inactive + Inconclusive)


In [316]:
# Step 6: Group CIDs by same outcome pattern
# Each row = one unique pattern, with count and list of CIDs sharing it
pattern_groups = (
    compound_matrix.groupby(["pattern", "behavior"])["PUBCHEM_CID"]
    .agg(num_compounds="count", cids=list)
    .reset_index()
    .sort_values("num_compounds", ascending=False)
    .reset_index(drop=True)
)

pattern_groups.to_csv(
    "/Users/yaminichitikela/Documents/casepase/Casepase-activation/results/all_cell_line_comparisions/pattern_groups_caspase_all.csv",
    index=False
)
print(f"Total unique patterns : {len(pattern_groups)}")
print(f"Total CIDs covered    : {pattern_groups['num_compounds'].sum()}")
print(f"\nPatterns by behavior:")
print(pattern_groups.groupby("behavior")["num_compounds"].agg(["count","sum"]).rename(columns={"count":"num_patterns","sum":"num_cids"}))
pattern_groups

Total unique patterns : 152
Total CIDs covered    : 7716

Patterns by behavior:
                                          num_patterns  num_cids
behavior                                                        
Active                                               1        66
Inactive                                            15      6967
Inconclusive                                         1        67
Mixed (Active + Inactive + Inconclusive)            69        77
Mixed (Active + Inactive)                           11       110
Mixed (Active + Inconclusive)                        4        70
Mixed (Inactive + Inconclusive)                     51       359


,pattern,behavior,num_compounds,cids
0,AID_1346980=Inactive | combined_hepg2_casepase...,Inactive,5897,"[4, 33, 34, 72, 79, 107, 135, 176, 178, 179, 1..."
1,AID_1346980=Inactive | AID_655=Inactive | AID_...,Inactive,1026,"[11, 13, 51, 66, 174, 180, 196, 227, 240, 241,..."
2,AID_1346980=Inactive | combined_hepg2_casepase...,Mixed (Inactive + Inconclusive),144,"[6, 727, 2157, 2377, 2448, 2801, 3151, 3627, 3..."
3,AID_1346980=Inconclusive | combined_hepg2_case...,Inconclusive,67,"[2126, 2351, 2726, 3042, 3225, 3372, 3386, 339..."
4,AID_1346980=Active | combined_hepg2_casepase.c...,Active,66,"[2247, 2799, 3191, 4807, 5154, 5564, 6928, 839..."
...,...,...,...,...
147,AID_1346980=Inactive | AID_655=Inactive | AID_...,Mixed (Inactive + Inconclusive),1,[447043]
148,AID_1346980=Inactive | AID_655=Inactive | AID_...,Inactive,1,[8363]
149,AID_1346980=Inactive | AID_655=Inactive | AID_...,Mixed (Inactive + Inconclusive),1,[8805]
150,AID_1346980=Inactive | AID_655=Inactive | AID_...,Mixed (Inactive + Inconclusive),1,[13101]


In [317]:
# Filter rows where PUBCHEM_CID is 299
result = consistent_groups[consistent_groups["PUBCHEM_CID"] == 299]

# Display the result
result

,PUBCHEM_CID,AID_1346980,AID_655,AID_656,AID_657,AID_658,AID_659,AID_660,AID_661,AID_663,AID_664,AID_665,AID_666,AID_667,combined_hepg2_casepase.csv,pattern,behavior
7703,299,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Inconclusive,Active,Inconclusive,Inactive,Inconclusive,Active,AID_1346980=Inconclusive | AID_655=Inconclusiv...,Mixed (Active + Inactive + Inconclusive)
